# Working with Time Series Data

Now that we understand datetime fundamentals (notebook 02), it's time to work with **real time series datasets**. In this notebook we will:

1. Learn the standard pattern for loading time series from CSV files
2. Explore several classic datasets, each illustrating different time series characteristics
3. Practice visual pattern recognition — identifying trend, seasonality, and noise by eye
4. Understand the difference between additive and multiplicative composition
5. Introduce the endogenous vs. exogenous distinction that appears throughout `statsmodels`

This maps to **Step 3 of the forecasting workflow**: *always start by graphing the data!*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100

DATA_DIR = Path("../data")

print(f"pandas  {pd.__version__}")
print(f"numpy   {np.__version__}")

---
## 1. Loading Time Series from CSV

The standard recipe for loading a time series CSV into pandas is:

1. **Load** with `pd.read_csv()`
2. **Parse** the date column (`parse_dates=True` or specify columns)
3. **Set** the date column as the index (`index_col=`)
4. **Verify** the frequency (`df.index.freq`)

Let's walk through this with the classic **Airline Passengers** dataset.

In [ ]:
# Step 1-3: Load, parse dates, and set index in one call
df_air = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    parse_dates=True,
    index_col="Month",
)

df_air.head()

In [ ]:
df_air.tail()

In [ ]:
df_air.info()

In [ ]:
df_air.describe()

### Inspecting the DatetimeIndex

A well-formed time series in pandas has a `DatetimeIndex` with a known **frequency**. Many `statsmodels` functions (decomposition, ARIMA, etc.) will raise errors or warnings if the frequency is not set.

In [ ]:
print(f"Index type : {type(df_air.index)}")
print(f"Index dtype: {df_air.index.dtype}")
print(f"Frequency  : {df_air.index.freq}")

If `freq` is `None`, we need to set it explicitly. `pd.infer_freq()` examines the spacing between index values and returns a frequency string (e.g., `'MS'` for month-start, `'D'` for daily).

In [ ]:
# Step 4: Set the frequency explicitly
if df_air.index.freq is None:
    inferred = pd.infer_freq(df_air.index)
    print(f"Inferred frequency: {inferred}")
    df_air.index.freq = inferred

print(f"Frequency is now: {df_air.index.freq}")

> **Why does frequency matter?** Many time series models need to know the spacing of observations. For example, seasonal decomposition needs to know how many observations make up one "season" (12 for monthly data with annual seasonality). If the frequency is missing, `statsmodels` has to guess — or it simply refuses to run.

---
## 2. Exploring Multiple Datasets

Let's load several classic time series datasets. Each one illustrates different combinations of **trend**, **seasonality**, and **noise**. For each dataset we will:

1. Load and inspect the structure
2. Plot the series
3. Note what components are visible

### 2a. Airline Passengers

Monthly totals of international airline passengers, 1949–1960. This is the **"hello world" of time series analysis**, originally published by Box & Jenkins.

Look for:
- **Upward trend**: passenger numbers grow over time
- **Growing seasonal amplitude**: the seasonal swings get larger as the level increases
- This growing amplitude is the hallmark of **multiplicative seasonality**

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_air, color='steelblue', linewidth=1.2)
ax.set_title("Airline Passengers (1949–1960)", fontsize=14)
ax.set_xlabel("Date")
ax.set_ylabel("Thousands of Passengers")
plt.tight_layout()
plt.show()

Notice how the peaks and troughs widen over time. The seasonal variation in 1960 is much larger than in 1949. This is **multiplicative** behavior — the seasonal effect is proportional to the level of the series.

### 2b. CO2 Concentrations (Mauna Loa)

Monthly atmospheric CO2 readings from the Mauna Loa Observatory in Hawaii, beginning in 1958. This is one of the most important scientific datasets ever collected.

Look for:
- **Strong upward trend**: CO2 levels are rising
- **Clear seasonal pattern**: the annual vegetation cycle (plants absorb CO2 in spring/summer)
- The seasonal amplitude stays roughly constant → closer to **additive** seasonality

In [ ]:
# This CSV uses separate year/month columns and has some missing values
df_co2 = pd.read_csv(DATA_DIR / "co2_mm_mlo.csv")
df_co2.head(10)

In [ ]:
# Build a proper datetime index from year + month columns
df_co2["date"] = pd.to_datetime(
    df_co2[["year", "month"]].assign(day=1)
)
df_co2 = df_co2.set_index("date")

# The 'interpolated' column fills gaps; use it for a complete series
df_co2 = df_co2[["interpolated"]].rename(columns={"interpolated": "CO2_ppm"})
df_co2.index.freq = pd.infer_freq(df_co2.index)

print(f"Shape: {df_co2.shape}")
print(f"Index freq: {df_co2.index.freq}")
df_co2.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_co2, color='seagreen', linewidth=1.0)
ax.set_title("Mauna Loa CO2 Concentrations", fontsize=14)
ax.set_xlabel("Date")
ax.set_ylabel("CO2 (ppm)")
plt.tight_layout()
plt.show()

The "sawtooth" pattern within each year is the seasonal cycle: CO2 drops when Northern Hemisphere vegetation grows in spring/summer and rises when leaves fall in autumn/winter.

### 2c. US Energy Production

Monthly index of US energy production. Another series with both trend and seasonality.

In [ ]:
df_energy = pd.read_csv(
    DATA_DIR / "EnergyProduction.csv",
    parse_dates=True,
    index_col="DATE",
)

df_energy.index.freq = pd.infer_freq(df_energy.index)
print(f"Shape: {df_energy.shape}")
print(f"Index freq: {df_energy.index.freq}")
df_energy.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_energy, color='darkorange', linewidth=1.0)
ax.set_title("US Energy Production Index", fontsize=14)
ax.set_xlabel("Date")
ax.set_ylabel("Energy Index")
plt.tight_layout()
plt.show()

Energy production shows a clear **upward trend** and **strong seasonal pattern** (energy demand peaks in summer for cooling and winter for heating). Unlike airline passengers, the seasonal swings stay relatively constant in size — this is closer to **additive** seasonality.

### 2d. Daily Female Births

Daily total female births in California, 1959. A full year of daily data — 365 observations.

In [ ]:
df_births = pd.read_csv(
    DATA_DIR / "DailyTotalFemaleBirths.csv",
    parse_dates=True,
    index_col="Date",
)

df_births.index.freq = pd.infer_freq(df_births.index)
print(f"Shape: {df_births.shape}")
print(f"Index freq: {df_births.index.freq}")
df_births.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_births, color='indianred', linewidth=0.8)
ax.set_title("Daily Total Female Births (1959)", fontsize=14)
ax.set_xlabel("Date")
ax.set_ylabel("Number of Births")
plt.tight_layout()
plt.show()

This series is a useful contrast to the others: **no obvious trend**, **no clear seasonality** — it is dominated by **noise** (random day-to-day variation). Not every time series has trend and seasonality; some are essentially stationary.

### 2e. US Population

Monthly US population estimates.

In [ ]:
df_pop = pd.read_csv(
    DATA_DIR / "uspopulation.csv",
    parse_dates=True,
    index_col="DATE",
)

df_pop.index.freq = pd.infer_freq(df_pop.index)
print(f"Shape: {df_pop.shape}")
print(f"Index freq: {df_pop.index.freq}")
df_pop.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_pop, color='mediumpurple', linewidth=1.5)
ax.set_title("US Population Estimate", fontsize=14)
ax.set_xlabel("Date")
ax.set_ylabel("Population (thousands)")
plt.tight_layout()
plt.show()

US population is almost a **pure trend** series — strong, nearly linear growth with **no seasonality** and very little noise. This kind of series is well-suited to simple trend models (even a straight line fits well).

---
## 3. Visual Pattern Recognition

Let's put four of these datasets side-by-side to sharpen our ability to identify time series components at a glance. This is arguably the most important skill in time series analysis: **always plot your data first**.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Airline Passengers
ax = axes[0, 0]
ax.plot(df_air, color='steelblue', linewidth=1.0)
ax.set_title("Airline Passengers", fontsize=12, fontweight='bold')
ax.set_ylabel("Thousands")
ax.text(
    0.02, 0.95,
    "Trend: Yes (up)\nSeasonality: Yes (12-mo)\nGrowing variance: Yes\n→ Multiplicative",
    transform=ax.transAxes, fontsize=8, verticalalignment='top',
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8),
)

# Energy Production
ax = axes[0, 1]
ax.plot(df_energy, color='darkorange', linewidth=1.0)
ax.set_title("Energy Production", fontsize=12, fontweight='bold')
ax.set_ylabel("Index")
ax.text(
    0.02, 0.95,
    "Trend: Yes (up)\nSeasonality: Yes (12-mo)\nGrowing variance: No\n→ Additive",
    transform=ax.transAxes, fontsize=8, verticalalignment='top',
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8),
)

# Daily Births
ax = axes[1, 0]
ax.plot(df_births, color='indianred', linewidth=0.8)
ax.set_title("Daily Female Births", fontsize=12, fontweight='bold')
ax.set_ylabel("Births")
ax.text(
    0.02, 0.95,
    "Trend: No\nSeasonality: No\nNoise: High\n→ Stationary / random",
    transform=ax.transAxes, fontsize=8, verticalalignment='top',
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8),
)

# US Population
ax = axes[1, 1]
ax.plot(df_pop, color='mediumpurple', linewidth=1.5)
ax.set_title("US Population", fontsize=12, fontweight='bold')
ax.set_ylabel("Population (thousands)")
ax.text(
    0.02, 0.95,
    "Trend: Yes (strong, up)\nSeasonality: No\nNoise: Very low\n→ Pure trend",
    transform=ax.transAxes, fontsize=8, verticalalignment='top',
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8),
)

for ax in axes.flat:
    ax.set_xlabel("Date")

fig.suptitle(
    "Visual Pattern Recognition: What Components Do You See?",
    fontsize=14, fontweight='bold', y=1.02,
)
plt.tight_layout()
plt.show()

**Key takeaway:** every time series is different. Some have trend, some have seasonality, some have both, some have neither. Your choice of forecasting model should be guided by what you see in the plot:

| Dataset | Trend | Seasonality | Noise | Character |
|:--------|:-----:|:-----------:|:-----:|:----------|
| Airline Passengers | Strong (up) | Yes (12-month) | Low | Multiplicative |
| Energy Production | Moderate (up) | Yes (12-month) | Moderate | Additive |
| Daily Births | None | None | High | Stationary |
| US Population | Very strong (up) | None | Very low | Pure trend |

---
## 4. Additive vs. Multiplicative Composition

In notebook 01 we introduced the idea that a time series $y_t$ can be decomposed into components. There are two main ways these components can combine:

### Additive Model

$$y_t = T_t + S_t + R_t$$

The components **add up**. The seasonal fluctuation has a roughly constant absolute size regardless of the level of the series.

### Multiplicative Model

$$y_t = T_t \times S_t \times R_t$$

The components **multiply**. The seasonal fluctuation is proportional to the level — as the trend rises, the seasonal swings get wider.

### The Visual Test

The simplest way to tell them apart:

> **If the seasonal variation grows with the level of the series → multiplicative.** 
> **If the seasonal variation stays constant → additive.**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Multiplicative example: Airline Passengers
ax = axes[0]
ax.plot(df_air, color='steelblue', linewidth=1.0)
ax.set_title("Multiplicative: Airline Passengers", fontsize=12, fontweight='bold')
ax.set_ylabel("Thousands of Passengers")
ax.set_xlabel("Date")
# Annotate the growing amplitude
ax.annotate(
    "Small swings", xy=(pd.Timestamp('1950-07'), 170),
    fontsize=9, color='red', fontweight='bold',
)
ax.annotate(
    "Large swings", xy=(pd.Timestamp('1959-01'), 480),
    fontsize=9, color='red', fontweight='bold',
)

# Additive example: Energy Production
ax = axes[1]
ax.plot(df_energy, color='darkorange', linewidth=1.0)
ax.set_title("Additive: Energy Production", fontsize=12, fontweight='bold')
ax.set_ylabel("Energy Index")
ax.set_xlabel("Date")
ax.annotate(
    "Similar swing size throughout",
    xy=(pd.Timestamp('1990-01'), 60),
    fontsize=9, color='red', fontweight='bold',
)

plt.tight_layout()
plt.show()

### The Log Transform Trick

A useful property: taking the **logarithm** of a multiplicative series converts it to additive form.

If $y_t = T_t \times S_t \times R_t$, then:

$$\log(y_t) = \log(T_t) + \log(S_t) + \log(R_t)$$

This is why you will often see analysts apply `np.log()` to series like airline passengers before modeling. It stabilizes the variance and makes the seasonal pattern constant in size.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(df_air, color='steelblue', linewidth=1.0)
axes[0].set_title("Original (Multiplicative)", fontsize=12)
axes[0].set_ylabel("Passengers")
axes[0].set_xlabel("Date")

axes[1].plot(np.log(df_air), color='steelblue', linewidth=1.0)
axes[1].set_title("Log-Transformed (now Additive)", fontsize=12)
axes[1].set_ylabel("log(Passengers)")
axes[1].set_xlabel("Date")

plt.tight_layout()
plt.show()

After the log transform, the seasonal swings are roughly constant — the multiplicative pattern has been converted to additive. This is a practical technique you will use repeatedly.

---
## 5. Endogenous vs. Exogenous Variables

Throughout `statsmodels` you will encounter two terms that can be confusing at first:

| Term | Meaning | In code |
|:-----|:--------|:--------|
| **Endogenous** (y) | The variable we are trying to **forecast** | `endog` |
| **Exogenous** (x) | External variables that may help **explain** or **predict** y | `exog` |

### Example

Suppose you want to forecast **daily electricity demand** for a utility company:

- **Endogenous variable**: electricity demand (megawatt-hours) — this is what we want to predict
- **Exogenous variables**: temperature, day of week, holiday indicator — these are external factors that influence demand but are not themselves being forecasted by our model

### Why does this matter?

- Simple models like **ARIMA** (Chapter 08) use only the endogenous variable — they forecast purely from the series' own past values
- More advanced models like **SARIMAX** (Chapter 09) can incorporate exogenous variables to improve forecasts
- When you see `model.fit(endog=y, exog=X)` in statsmodels, now you know: `y` is what we predict, `X` is the external information we use to help

In [ ]:
# A concrete illustration of the distinction
print("In all the datasets above, we have a single endogenous variable:")
print()
datasets = {
    "Airline Passengers": ("Thousands of Passengers", "endog"),
    "CO2 Concentrations": ("CO2 ppm", "endog"),
    "Energy Production": ("Energy Index", "endog"),
    "Daily Female Births": ("Number of Births", "endog"),
    "US Population": ("Population Estimate", "endog"),
}

for name, (variable, role) in datasets.items():
    print(f"  {name:25s} → {variable:30s} [{role}]")

print()
print("None of these datasets include exogenous variables.")
print("We'll encounter exogenous variables when we reach SARIMAX in Chapter 09.")

---
## 6. Summary and What's Next

In this notebook we have:

- Learned the standard **load → parse → index → verify frequency** pattern for CSV files
- Explored **five real datasets**, each with different characteristics
- Practiced **visual pattern recognition** — identifying trend, seasonality, and noise from plots
- Distinguished **additive** from **multiplicative** composition (and the log-transform trick)
- Introduced the **endogenous vs. exogenous** distinction used throughout `statsmodels`

### Coming up

| Chapter | Topic |
|:--------|:------|
| **Ch 02** | Pandas time series operations — resampling, shifting, rolling windows |
| **Ch 03** | Formal visualization techniques — seasonal plots, lag plots, ACF/PACF |
| **Ch 04** | Time series decomposition — separating trend, seasonality, and residuals mathematically |